# 기업 리뷰와 채용공고를 결합한 지원 의사결정 점수

**작성자:** bolly0809-coder  
**프로젝트:** 취준생 장기재직 지원 서비스 팀 프로젝트  
**공개본 점검일:** 2026-09-15

기업 리뷰 원문을 정제하고 7개 영역의 긍정·부정 신호로 구조화한 뒤, 기업별 리뷰 점수와 채용공고 점수를 결합한 과정을 정리했다. 이 노트북은 제가 맡은 **리뷰 전처리·사전 구축·점수화·통합 테이블 설계**를 중심으로 재구성한 공개용 분석 기록이다.

> 원문 리뷰, 기업별 상세 결과, 크롤링 세션, 데이터베이스와 팀 내부 파일은 공개하지 않는다. 아래 수치는 최종 제출 자료와 산출물을 교차 확인한 집계 결과다.

## 1. 분석 질문과 데이터 범위

1. 비정형 리뷰를 기업 간 비교가 가능한 지표로 바꿀 수 있는가?
2. 리뷰 수가 적은 기업의 점수가 과도하게 높거나 낮아지는 문제를 어떻게 줄일 것인가?
3. 구직자의 선호와 회피 요인을 점수에 어떻게 반영할 것인가?
4. 리뷰 점수와 채용공고 점수를 어떤 기준으로 결합할 것인가?

| 단계 | 확인한 규모 |
|---|---:|
| 프로젝트 보고서 기준 수집 리뷰 | 29,680건 |
| 최종 리뷰 점수화 입력 | 21,197건 |
| 리뷰 집계 기업 | 433개 |
| 리뷰가 연결된 채용공고 | 515건 |
| 통합 테이블 기업 | 383개 |
| 대상 직무 | 데이터 분석가 176건, 백엔드 개발자 339건 |

In [1]:
import re
import numpy as np
import pandas as pd

PROJECT_COUNTS = pd.Series({
    "수집 리뷰": 29_680,
    "점수화 입력 리뷰": 21_197,
    "리뷰 집계 기업": 433,
    "통합 채용공고": 515,
    "통합 기업": 383,
}, name="건수")

print(PROJECT_COUNTS.to_string())

수집 리뷰        29680
점수화 입력 리뷰    21197
리뷰 집계 기업        433
통합 채용공고         515
통합 기업            383


## 2. 분석 흐름

```mermaid
flowchart LR
    A[기업 리뷰 원문] --> B[중복·결측·문자 정제]
    B --> C[표준화·복합명사·불용어 사전]
    C --> D[Kiwi 토큰화와 문장 분리]
    D --> E[7개 영역 긍정·부정 신호]
    E --> F[기업별 집계와 리뷰 수 신뢰도]
    F --> G[평점·텍스트·감성 점수]
    G --> H[사용자 선호·회피·업무방식 반영]
    H --> I[기업 리뷰 등급]
    I --> J[채용공고 40% + 리뷰 60%]
    J --> K[지원 우선순위 등급]
```

분석 단위는 리뷰 한 건에서 기업으로, 마지막에는 채용공고 한 건으로 바뀐다. 각 단계에서 단위가 달라지므로 기업명을 정규화하고 리뷰 수를 별도 보존했다.

## 3. 담당 범위

- 기업 리뷰 4차 정제본의 중복과 결측 상태 확인
- 기업명 정규화와 채용공고 테이블 연결 키 구성
- 표준화 사전, 복합명사 사전, 카테고리 사전, 수동 불용어 정비
- Kiwi 기반 문장 분리와 토큰화
- 복지·연봉·성장·워라밸·조직문화·안정성·직무적합 7개 영역의 긍정·부정 신호 집계
- 리뷰 수에 따른 신뢰도 보정과 사용자 응답 기반 점수 설계
- 리뷰 점수와 채용공고 점수를 결합한 최종 분석 테이블 구축

In [2]:
def normalize_company(name: str) -> str:
    '''채용공고와 리뷰 테이블을 연결하기 위한 최소 기업명 정규화.'''
    name = re.sub(r"\(주\)|㈜|주식회사\s*|\(유\)|\(재\)", "", str(name))
    return name.strip()


def review_reliability(review_count: int) -> float:
    '''리뷰 수가 적을수록 전체 평균 쪽으로 더 강하게 보정한다.'''
    if review_count >= 30:
        return 1.0
    if review_count >= 10:
        return 0.8
    if review_count >= 3:
        return 0.6
    return 0.3


reliability_demo = pd.DataFrame({"review_count": [1, 3, 10, 30]})
reliability_demo["reliability"] = reliability_demo["review_count"].map(review_reliability)
print(reliability_demo.to_string(index=False))

 review_count  reliability
            1          0.3
            3          0.6
           10          0.8
           30          1.0


## 4. 사전과 영역 신호

| 사전 | 최종 규모 | 용도 |
|---|---:|---|
| 표준화 규칙 | 194개 | 맞춤법·표기 변형과 복합명사를 같은 표현으로 통일 |
| 복합명사 | 267개 | 의미가 분리되면 안 되는 표현을 하나의 토큰으로 등록 |
| 카테고리 단어 | 596개 | 토큰을 7개 분석 영역에 연결 |
| 수동 불용어 | 177개 | 분석 의미가 낮거나 반복되는 표현 제외 |
| 전체 불용어 | 634개 | 수동 목록에 기업명·직무명 등을 더한 실행 시점 기준 |

`pros`와 `cons` 열만으로 긍정·부정을 단정하지 않고, 영역 토큰 주변 ±3개 토큰에서 서술어를 확인했다. 주변 서술어가 없으면 원문 열의 장점·단점 구분을 보조 기준으로 사용했다.

In [3]:
CATEGORY_COLUMNS = {
    "복지": "welfare",
    "연봉": "salary",
    "성장": "growth",
    "워라밸": "worklife",
    "조직문화": "culture",
    "안정성": "stability",
    "직무적합": "jobfit",
}


def clamp(score: float, low: float = 0, high: float = 100) -> float:
    return max(low, min(high, score))


def category_text_score(positive_rate: float, negative_rate: float) -> float:
    '''55점을 기준으로 긍정 비율은 +60, 부정 비율은 -70을 적용.'''
    return round(clamp(55 + positive_rate * 60 - negative_rate * 70), 1)


signal_weights = pd.Series({
    "연봉": 0.18,
    "안정성": 0.18,
    "성장": 0.18,
    "워라밸": 0.18,
    "조직문화": 0.18,
    "복지": 0.10,
}, name="text_signal_weight")

print(signal_weights.to_string())
print(f"합계: {signal_weights.sum():.2f}")

연봉      0.18
안정성     0.18
성장      0.18
워라밸     0.18
조직문화    0.18
복지      0.10
합계: 1.00


## 5. 리뷰 점수 구조

평점 기반 점수는 5점 척도를 100점으로 환산한다. 텍스트 신호는 사전 오탐 가능성을 고려해 기본 리뷰 점수의 20%만 반영했다. 별도 긍정·부정 감성 단어 점수에는 10%를 배정했다.

사용자 응답은 다음 세 질문으로 구성했다.

- Q1: 직장 선택에서 가장 중요한 기준
- Q2: 이직을 고민하게 되는 주된 이유
- Q3: 혼자 집중하는 방식과 협업 방식 중 선호 유형

기본 리뷰 점수와 세 응답 점수를 결합한 뒤, 리뷰 수가 적은 기업은 전체 평균 쪽으로 축소한다.

In [4]:
RATING_WEIGHTS = {
    "overall_score": 0.15,
    "welfare_score": 0.10,
    "worklife_score": 0.15,
    "culture_score": 0.15,
    "promotion_score": 0.15,
    "management_score": 0.15,
    "rate_recommend": 0.10,
    "rate_growth": 0.05,
}


def weighted_score(row: pd.Series, weights: dict[str, float]) -> float:
    return sum(float(row[column]) * weight for column, weight in weights.items())


def base_review_score(rating_score: float, text_signal: float, sentiment: float) -> float:
    return round(rating_score * 0.70 + text_signal * 0.20 + sentiment * 0.10, 1)


def personalized_raw_score(base: float, q1: float, q2: float, q3: float) -> float:
    return round(base * 0.45 + q1 * 0.25 + q2 * 0.20 + q3 * 0.10, 1)


def reliability_adjust(raw_score: float, review_count: int, global_mean: float) -> float:
    reliability = review_reliability(review_count)
    return round(raw_score * reliability + global_mean * (1 - reliability), 1)


review_score_weights = pd.Series({
    "평점 기반": 0.70,
    "영역 텍스트 신호": 0.20,
    "전체 감성 신호": 0.10,
}, name="base_review_score")
personal_weights = pd.Series({
    "기본 리뷰 점수": 0.45,
    "Q1 선호": 0.25,
    "Q2 회피 위험": 0.20,
    "Q3 업무방식": 0.10,
}, name="review_raw_score")

print(pd.concat([review_score_weights, personal_weights], axis=1).fillna("-").to_string())

          base_review_score review_raw_score
평점 기반                  0.7                -
영역 텍스트 신호             0.2                -
전체 감성 신호               0.1                -
기본 리뷰 점수                 -             0.45
Q1 선호                       -             0.25
Q2 회피 위험                   -              0.2
Q3 업무방식                    -              0.1


In [5]:
def percentile_display_score(percentile: float) -> float:
    '''기업 간 상대 위치를 40~90점의 서비스 표시 점수로 변환.'''
    if percentile < 0.80:
        return round(clamp(40 + (percentile / 0.80) * 40, 40, 80), 1)
    return round(clamp(80 + ((percentile - 0.80) / 0.20) * 10, 80, 90), 1)


def review_grade(display_score: float, review_count: int) -> str:
    '''리뷰 5건 미만 기업은 최고 등급을 조건부 추천으로 제한.'''
    if review_count < 5:
        return "조건부 추천" if display_score >= 55 else "비추천"
    if display_score >= 75:
        return "추천"
    if display_score >= 55:
        return "조건부 추천"
    return "비추천"


grade_examples = pd.DataFrame({
    "percentile": [0.10, 0.50, 0.90],
    "review_count": [20, 20, 3],
})
grade_examples["display_score"] = grade_examples["percentile"].map(percentile_display_score)
grade_examples["grade"] = grade_examples.apply(
    lambda row: review_grade(row["display_score"], int(row["review_count"])), axis=1
)
print(grade_examples.to_string(index=False))

 percentile  review_count  display_score  grade
        0.1            20           45.0    비추천
        0.5            20           65.0 조건부 추천
        0.9             3           85.0 조건부 추천


표시 점수는 기업 간 **상대적 위치**를 보여준다. 80번째 백분위 미만은 40~80점, 상위 20%는 80~90점으로 변환했다. 따라서 80점이 절대적인 기업 품질이나 장기재직 확률 80%를 뜻하지 않는다.

In [6]:
review_results = pd.DataFrame({
    "항목": ["분석 리뷰", "기업", "평균 표시 점수", "중앙 표시 점수", "추천", "조건부 추천", "비추천"],
    "값": [21_197, 433, 65.1, 64.9, 129, 176, 128],
})

print(review_results.to_string(index=False))

       항목       값
    분석 리뷰  21197.0
        기업    433.0
 평균 표시 점수     65.1
 중앙 표시 점수     64.9
        추천    129.0
    조건부 추천    176.0
       비추천    128.0


## 6. 채용공고 점수와 결합

리뷰가 연결된 공고만 사용했다. 공고의 직무 적합도·정보 품질·위험 신호를 반영한 점수에 40%, 리뷰 점수에 60%를 배정했다.

`final_score = posting_final_score × 0.40 + review_final_score × 0.60`

통합 점수가 72점 이상이면 추천, 60점 이상이면 조건부 추천, 60점 미만이면 비추천으로 구분했다. 72점은 통합 점수의 상위 약 25% 구간을 참고한 운영 기준이며, 외부 성과 데이터로 최적화한 임계값은 아니다.

In [7]:
def integrated_score(posting_score: float, review_score: float) -> float:
    return round(posting_score * 0.40 + review_score * 0.60, 1)


def integrated_grade(score: float) -> str:
    if score >= 72:
        return "추천"
    if score >= 60:
        return "조건부 추천"
    return "비추천"


integration_results = pd.DataFrame({
    "항목": ["통합 공고", "통합 기업", "평균 통합 점수", "중앙 통합 점수", "추천", "조건부 추천", "비추천"],
    "값": [515, 383, 66.6, 67.0, 124, 255, 136],
})

print(integration_results.to_string(index=False))

       항목      값
    통합 공고   515.0
    통합 기업   383.0
 평균 통합 점수    66.6
 중앙 통합 점수    67.0
        추천   124.0
    조건부 추천   255.0
       비추천   136.0


## 7. 검증에서 확인한 중요한 문제

최종 제출 자료에는 랜덤 포레스트 정확도 0.8 이상이 제시되어 있다. 다만 머신러닝용 `final_label`은 통합 점수의 등급에서 만들었고, 입력 특성에는 그 등급을 직접 결정하는 `posting_final_score`와 `review_final_score`가 포함됐다. 두 점수를 40:60으로 합치면 타깃 등급을 거의 복원할 수 있으므로, 해당 정확도를 새로운 장기재직 결과에 대한 예측 성능으로 해석하기 어렵다.

이 공개본에서는 머신러닝 정확도를 핵심 성과에서 제외하고, **규칙 기반 점수 설계와 데이터 통합 과정**을 분석 결과로 제시한다. 실제 예측 모델로 발전시키려면 입사 여부와 재직기간 같은 독립적인 결과 라벨을 확보하고, 시간 순서에 따른 별도 평가 자료에서 검증해야 한다.

## 8. 한계와 다음 단계

- 리뷰 작성자는 전체 재직자를 대표하지 않을 수 있고, 기업별 리뷰 수와 작성 시점도 다르다.
- 사전 기반 영역 분류와 주변 서술어 규칙의 정밀도·재현율을 별도의 수작업 라벨 표본으로 검증하지 못했다.
- 백분위 표시 점수, 질문 가중치, 40:60 통합 비율과 등급 임계값은 서비스 운영 규칙이다. 장기재직이나 지원 전환에 대한 인과효과를 의미하지 않는다.
- 실제 클릭·저장·지원·입사·재직기간 데이터가 없어 추천 결과의 효과를 측정하지 못했다.
- 실시간 채용공고 API를 연결하지 못해 제출 시점의 정적 데이터로 분석했다.

다음 분석에서는 리뷰 문장 표본을 직접 라벨링해 영역·감성 규칙을 검증하고, 데이터 수집 시점을 저장해 최신성을 관리해야 한다. 실제 사용자 행동과 재직 결과가 쌓이면 점수 규칙과 예측 모델을 분리해 평가할 수 있다.

## 9. 공개 범위와 재실행

이 노트북은 공개 가능한 코드 구조와 집계 수치만 담은 재구성본이다. 원문 리뷰, 기업별 상세 점수, 사전 파일, 크롤링 세션, SQLite 데이터베이스, 팀 내부 제출본은 포함하지 않는다.

전체 파이프라인을 다시 실행하려면 원본 리뷰 스키마, 세 종류의 사용자 사전, 수동 불용어 파일, Kiwi 형태소 분석기와 채용공고 점수 테이블이 필요하다. 공개본은 데이터 라이선스와 리뷰 작성자 보호를 위해 이 입력 파일들을 제공하지 않는다.